# This is the workflow of SDM

In [ ]:
import os
import geopandas as gpd
import numpy as np
import libpysal
from spreg import ML_Lag
from libpysal.weights import DistanceBand, lag_spatial
import re
import pandas as pd 
# -----------------------------
# 设置路径
# -----------------------------
grid_folder = r"E:\seoul\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

# -----------------------------
# 准备变量
# -----------------------------
year = 2016
for year in [2016,2023]:
    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    explanatory_vars_clean = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR']
    grid_folder = r'E:\seoul\480_based'
    output_file = r'E:\seoul\480_based\statistics\SDM_effects.xlsx'
    def star(p):
        return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '.' if p < 0.1 else ''

    # 使用 ExcelWriter，mode='a' 可以追加 sheet（如果文件存在）
    with pd.ExcelWriter(output_file, engine='openpyxl', mode='a' if os.path.exists(output_file) else 'w',
                        if_sheet_exists='replace') as writer:

        for filename in os.listdir(grid_folder):
            if filename.endswith(f'city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp'):
                path = os.path.join(grid_folder, filename)
                gdf = gpd.read_file(path)
                gdf = gdf.replace([np.inf, -np.inf], np.nan)

                for target in target_vars:
                    if target not in gdf.columns:
                        continue

                    # 只保留完整数据
                    data = gdf[explanatory_vars + [target]].dropna()
                    data_gdf = gdf.loc[data.index].reset_index(drop=True)

                    threshold = 1000
                    w = DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                    w.transform = 'r'  # 行标准化
                    W = np.array(w.full()[0])  # 稠密矩阵
                    # print(data.shape)
                    
                    # -----------------------------
                    # y 与 X
                    # -----------------------------
                    yi = data[target].values.reshape(-1, 1)
                    X_main = data[explanatory_vars].values
                    X_all = X_main.copy()
                    name_x = explanatory_vars

                    # -----------------------------
                    # SDM 模型估计
                    # -----------------------------
                    slx_vars_bool = [v in explanatory_vars_clean for v in explanatory_vars]
                    model_sdm = ML_Lag(
                        yi, X_all, w=w, slx_lags=1, slx_vars=slx_vars_bool, # type: ignore
                        name_y=target, name_x=name_x,
                        name_w=f"W{threshold}", name_ds=f"yr{year}",
                        spat_diag=True, spat_impacts=['full'] # type: ignore
                    )

                    coefs = model_sdm.betas.flatten()
                    vars_ = model_sdm.name_x
                    print(model_sdm.rho)
                    print(coefs)
                    print(vars_)

                    # ==================================================
                    summary_str = str(model_sdm.summary)
                    lines = summary_str.split("\n")
                    impacts_data = []
                    capture = False
                    for line in lines:
                        if "SPATIAL DURBIN MODEL IMPACTS" in line:
                            capture = True
                            continue
                        if capture:
                            if not line.strip():  # 遇到空行就停
                                break
                            # 跳过分隔线和表头
                            if "Direct" in line and "Indirect" in line and "Total" in line:
                                continue
                            if set(line.strip()) == {"-"}:  # 全是横线
                                continue
                            parts = re.split(r"\s+", line.strip())
                            if len(parts) == 4:
                                var, direct, indirect, total = parts
                                impacts_data.append([var, float(direct), float(indirect), float(total)])

                    df_impacts = pd.DataFrame(impacts_data, columns=["Variable","Direct","Indirect","Total"])
                    print(df_impacts)

                    # # 👉 保存 Excel
                    # 保存到 Excel，不同 target 用 sheet 名
                    sheet_name = target
                    df_impacts.to_excel(writer, sheet_name=sheet_name, index=False)
                    print(f"Saved sheet: {sheet_name}")


# 验证参数和图像的关系 --- 通过直接的参数yhat = (1-rho*W)^-1 * (Constant + beta*X + theta*WX)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import lag_spatial

for feature in explanatory_vars:
    other_features = [f for f in explanatory_vars if f != feature]

    # 用原数据均值固定其他变量
    X_matrix = data[explanatory_vars].copy().values  # 原始数据行数
    predy_list = []

    feature_min = data[feature].min()
    feature_max = data[feature].max()
    n_points = 100
    feature_range = np.linspace(feature_min, feature_max, n_points)

    for val in feature_range:
        X_matrix_temp = X_matrix.copy()
        X_matrix_temp[:, explanatory_vars.index(feature)] = val  # 修改 BCR
        # WX_clean
        WX_clean = lag_spatial(w, X_matrix_temp[:, :len(explanatory_vars_clean)])
        # I - rho*W
        I = np.eye(len(X_matrix_temp))
        W_dense = np.array(w.full()[0])
        y_temp = np.linalg.inv(I - rho * W_dense) @ (constant + X_matrix_temp @ beta + WX_clean @ theta)
        predy_list.append(y_temp.mean())  # 取平均作为单变量效果

    plt.figure(figsize=(8,5))

    plt.scatter(data[feature], model_sdm.predy.flatten(), s=8, alpha=0.5, label='Original predy')
    plt.plot(feature_range, predy_list, color='red', linewidth=2, label=f'Total effect of {feature}')

    plt.xlabel(feature)
    plt.ylabel(f"Predicted {target}")
    plt.title(f"SDM Predicted Values vs {feature}")
    plt.grid(True)
    plt.ylim(-15,-1)
    plt.legend()
    plt.show()

# 验证参数和图像的关系 --- 通过excel 的参数

In [ ]:
effects_file = r"E:\seoul\480_based\statistics\SDM_all_params_480m_NOW.xlsx"
df_effects = pd.read_excel(effects_file)

df_effects = df_effects[df_effects['Target'] == 'hr_2016'].reset_index(drop=True)
row = df_effects.loc[0]

explanatory_vars = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
explanatory_vars_clean = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR']
beta = np.array([row[v] for v in explanatory_vars])
theta = np.array([row[f"W_{v}"] for v in explanatory_vars_clean])
constant = np.ones(len(data)) * row['CONSTANT']

X_main = data[explanatory_vars].values
WX_clean = lag_spatial(w, data[explanatory_vars_clean].values)
W_dense = np.array(w.full()[0])
I = np.eye(len(data))

y_final = np.linalg.inv(I - rho * W_dense) @ (
    constant + X_main @ beta + WX_clean @ theta
)

import numpy as np
import matplotlib.pyplot as plt
from libpysal.weights import lag_spatial

for feature in explanatory_vars:
    other_features = [f for f in explanatory_vars if f != feature]

    # 用原数据均值固定其他变量
    X_matrix = data[explanatory_vars].copy().values  # 原始数据行数
    predy_list = []

    feature_min = data[feature].min()
    feature_max = data[feature].max()
    n_points = 100
    feature_range = np.linspace(feature_min, feature_max, n_points)

    for val in feature_range:
        X_matrix_temp = X_matrix.copy()
        X_matrix_temp[:, explanatory_vars.index(feature)] = val  # 修改 BCR
        # WX_clean
        WX_clean = lag_spatial(w, X_matrix_temp[:, :len(explanatory_vars_clean)])
        # I - rho*W
        I = np.eye(len(X_matrix_temp))
        W_dense = np.array(w.full()[0])
        y_temp = np.linalg.inv(I - rho * W_dense) @ (constant + X_matrix_temp @ beta + WX_clean @ theta)
        predy_list.append(y_temp.mean())  # 取平均作为单变量效果

    plt.figure(figsize=(8,5))

    plt.scatter(data[feature], model_sdm.predy.flatten(), s=8, alpha=0.5, label='Original predy')
    plt.plot(feature_range, predy_list, color='red', linewidth=2, label=f'Total effect of {feature}')

    plt.xlabel(feature)
    plt.ylabel(f"Predicted {target}")
    plt.title(f"SDM Predicted Values vs {feature}")
    plt.grid(True)
    plt.ylim(-15,-1)
    plt.legend()
    plt.show()

# 方法2 - df_impacts 

In [ ]:
df_impacts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_effects = df_impacts
for feature in explanatory_vars:
    TE_total = df_effects.loc[df_effects['Variable'] == feature, 'Total'].values[0]

    # -----------------------------
    # y_hat_i ≈ y_mean + TE_total * (BCR_i - BCR_mean)
    # -----------------------------
    y_base_mean = data['hr_2016'].mean()       # 或者使用模型原始 y_base.mean()
    bcr_mean = data[feature].mean()
    bcr_values = data[feature].values

    yhat = y_base_mean + TE_total * (bcr_values - bcr_mean)

    plt.figure(figsize=(8,5))
    plt.scatter(bcr_values, data['hr_2016'], s=8, alpha=0.5, label='Original hr_2016')
    plt.plot(bcr_values, yhat,color='red',  alpha=0.5, label='Predicted y_hat (via TE)')
    plt.xlabel(feature)
    plt.ylabel('hr_2016')
    plt.title('Prediction using SDM Total Effect')
    plt.legend()
    plt.ylim(-15,-1)
    plt.grid(True)
    plt.show()


# ->>>>>> 直接跳转， 最终集大成代码

In [ ]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np
import libpysal
from spreg import ML_Error, ML_Lag
from libpysal.weights import DistanceBand, lag_spatial

# -----------------------------
# 设置路径
# -----------------------------
grid_folder = r"E:\seoul\480_based"
output_folder = os.path.join(grid_folder, 'statistics')
os.makedirs(output_folder, exist_ok=True)

results_list = []
def show_formula(model, model_type='SDM'):
    # 处理 y 名称
    y_name = model.name_y if isinstance(model.name_y, str) else model.name_y[0]

    coefs = model.betas.flatten()
    vars_ = model.name_x

    terms = []
    for coef, var in zip(coefs, vars_):
        if var.lower() in ['const', 'constant']:  # 常数项
            terms.append(f"{coef:.4f}")
        else:
            terms.append(f"{coef:.4f}*{var}")

    formula = f"{y_name} = "

    # SDM rho 处理
    if model_type == 'SDM' and hasattr(model, 'rho'):
        rho_term = f"{model.rho:.4f}*W{y_name}"
        formula += " + ".join(terms[:1] + [rho_term] + terms[1:])  # 常数项 0 + Wy + X + WX
        return formula

    formula += " + ".join(terms)
    return formula


param_list = []

for year in [2023, 2016]: 

    target_vars = [f'nor_{year}', f'ext_{year}', f'hr_{year}']
    explanatory_vars = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR', 'Dist_W', 'Dist_P', 'Dist_M']
    explanatory_vars_clean = ['BCR', 'BHV', 'SVF', 'NDVI', 'EV', 'WR']

    for filename in os.listdir(grid_folder):
        if filename.endswith(f'city{year}_lst_ratio_grid_480m_bcr_bhv_ndvi_svf_ev_distbp_distmt_distwb_wr_xy_clean.shp'):
            path = os.path.join(grid_folder, filename)
            gdf = gpd.read_file(path)
            gdf = gdf.replace([np.inf, -np.inf], np.nan)

            for target in target_vars:
                if target not in gdf.columns:
                    continue

                # 删除 NA
                data = gdf[explanatory_vars + [target]].dropna()
                data_gdf = gdf.loc[data.index]

                # 权重矩阵
                threshold = 1000
                w = DistanceBand.from_dataframe(data_gdf, threshold=threshold, binary=False)
                w.transform = 'r'
                yi = data[target].values.reshape(-1, 1)
                X_main = data[explanatory_vars].values

                # SDM slx 变量
                slx_vars_bool = [v in explanatory_vars_clean for v in explanatory_vars]

                # --- 模型 ---
                model_sdm = ML_Lag(
                    yi, X_main, w=w,
                    slx_lags=1, slx_vars=slx_vars_bool,
                    name_y=target, name_x=explanatory_vars,
                    name_w=f"W{threshold}", name_ds=f"yr{year}",
                    spat_diag=True, spat_impacts=['full']
                )

                # --- 提取系数 ---
                formula_sdm = show_formula(model_sdm, model_type='SDM')
                coef_sdm = dict(zip(model_sdm.name_x, model_sdm.betas.flatten()))

                param_list.append({
                    "Year": year,
                    "Grid": filename.split("_")[4],
                    "Target": target,
                    "Model": "SDM",
                    "Formula": formula_sdm,
                    **coef_sdm
                })

# -----------------------------
# 输出
# -----------------------------
all_params_df = pd.DataFrame(param_list)
param_output = os.path.join(output_folder, "SDM_all_params_480m_NOW.xlsx")
all_params_df.to_excel(param_output, index=False)

print("保存完成")